# 1장 2강: 좋은 지표의 조건 — 실습문제

## 실습 목표

- Ravenstack의 비즈니스 목표에 맞춰 KPI, 보조 지표, 허무 지표를 구분할 수 있다.
- 구독 데이터에서 현재 반복 매출과 이탈률을 계산할 수 있다.
- 요금제 또는 유입 경로별 지표를 비교하여 개선이 필요한 대상을 찾을 수 있다.
- 행동 가능성, 비교 가능성, 이해 용이성을 기준으로 지표의 적절성을 판단할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- `ravenstack_accounts.csv`
- `ravenstack_subscriptions.csv`

Ravenstack은 기업 고객에게 구독형 소프트웨어를 제공하는 B2B SaaS 서비스입니다.

이번 실습에서는 Ravenstack의 핵심 목표를 다음과 같이 가정합니다.

> **유료 구독을 안정적으로 유지하면서 반복 매출을 늘린다.**

주요 컬럼은 다음과 같습니다.

| 테이블 | 컬럼 | 의미 |
|---|---|---|
| accounts | `account_id` | 고객사 식별자 |
| accounts | `referral_source` | 고객사가 유입된 경로 |
| accounts | `signup_date` | 고객사 가입일 |
| accounts | `churn_flag` | 고객사 이탈 여부 |
| subscriptions | `subscription_id` | 구독 식별자 |
| subscriptions | `plan_tier` | 구독 요금제 |
| subscriptions | `mrr_amount` | 월간 반복 매출(MRR) |
| subscriptions | `is_trial` | 체험 구독 여부 |
| subscriptions | `end_date` | 구독 종료일 |
| subscriptions | `churn_flag` | 구독 이탈 여부 |

> 비율은 별도 지시가 없으면 소수점 둘째 자리의 백분율로 출력합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. 두 CSV 파일을 각각 `accounts`, `subscriptions`에 불러오세요.
3. 각 데이터의 행과 열 개수, 상위 5개 행을 확인하세요.
4. 분석 대상 컬럼의 자료형과 결측치 개수를 확인하세요.
5. `signup_date`, `start_date`, `end_date`를 날짜형으로 변환하세요.
6. `mrr_amount`의 기초 통계량과 `plan_tier`의 빈도를 확인하세요.

In [2]:
# 실습 준비 코드를 작성하세요.
import pandas as pd

accounts = pd.read_csv("C:/first-project/ravenstack_accounts.csv")
subscriptions = pd. read_csv("C:/first-project/ravenstack_subscriptions.csv")

accounts.head()


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


In [3]:
subscriptions.head()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True


In [4]:
print(accounts.dtypes)
print(accounts.isnull().sum())

account_id           str
account_name         str
industry             str
country              str
signup_date          str
referral_source      str
plan_tier            str
seats              int64
is_trial            bool
churn_flag          bool
dtype: object
account_id         0
account_name       0
industry           0
country            0
signup_date        0
referral_source    0
plan_tier          0
seats              0
is_trial           0
churn_flag         0
dtype: int64


In [5]:
print(subscriptions.dtypes)
print(subscriptions.isnull().sum())

subscription_id        str
account_id             str
start_date             str
end_date               str
plan_tier              str
seats                int64
mrr_amount           int64
arr_amount           int64
is_trial              bool
upgrade_flag          bool
downgrade_flag        bool
churn_flag            bool
billing_frequency      str
auto_renew_flag       bool
dtype: object
subscription_id         0
account_id              0
start_date              0
end_date             4514
plan_tier               0
seats                   0
mrr_amount              0
arr_amount              0
is_trial                0
upgrade_flag            0
downgrade_flag          0
churn_flag              0
billing_frequency       0
auto_renew_flag         0
dtype: int64


In [6]:
accounts['signup_date'] = pd.to_datetime(accounts['signup_date'])
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'])  # 결측치는 NaT로 유지됨

print(subscriptions['mrr_amount'].describe())
print(accounts['plan_tier'].value_counts())

count     5000.000000
mean      2267.749400
std       3421.375348
min          0.000000
25%        285.000000
50%        931.000000
75%       2786.000000
max      33830.000000
Name: mrr_amount, dtype: float64
plan_tier
Pro           178
Basic         168
Enterprise    154
Name: count, dtype: int64


---

## 필수 1. 비즈니스 목표에 맞는 지표 구분하기

### 문제 1-1. 현재 반복 매출을 중심으로 지표를 계산하고 분류하기

#### 문제 설명

Ravenstack은 **유료 구독을 안정적으로 유지하면서 반복 매출을 늘리는 것**을 핵심 목표로 정했습니다.

다음 세 지표를 직접 계산한 뒤 현재 목표를 기준으로 KPI, 보조 지표, 허무 지표로 분류하세요.

- 현재 유료 구독 MRR
- 전체 구독 이탈률
- 누적 가입 고객사 수

이번 실습에서는 `end_date`가 비어 있고 `is_trial`이 `False`인 구독을 **현재 활성 유료 구독**으로 정의합니다.

#### 요구사항

1. 현재 활성 유료 구독 조건을 불리언 변수 `active_paid_mask`로 만드세요.
2. 현재 활성 유료 구독의 `mrr_amount` 합계를 `current_paid_mrr`로 계산하세요.
3. `subscriptions`의 `churn_flag` 평균으로 `subscription_churn_rate`를 계산하세요.
4. `accounts`의 고유한 `account_id` 개수를 `cumulative_accounts`로 계산하세요.
5. 세 지표를 알아보기 쉬운 형식으로 출력하세요.
6. 현재 비즈니스 목표를 기준으로 다음과 같이 분류하고 이유를 설명하세요.
   - 현재 유료 구독 MRR: KPI
   - 전체 구독 이탈률: 보조 지표
   - 누적 가입 고객사 수: 허무 지표 후보

#### 해석 질문

**Q1.** 현재 유료 구독 MRR을 KPI로 볼 수 있는 이유는 무엇인가요?  
**Q2.** 전체 구독 이탈률은 현재 유료 구독 MRR을 이해하는 데 어떻게 도움을 주나요?  
**Q3.** 누적 가입 고객사 수만으로 현재 서비스가 성장한다고 판단하기 어려운 이유는 무엇인가요?

#### 제출 결과

- 세 지표의 계산 코드와 결과
- KPI, 보조 지표, 허무 지표 분류
- 분류 이유
- Q1~Q3 답변

In [7]:
# 필수 1 코드를 작성하세요.
import pandas as pd

accounts = pd.read_csv('ravenstack_accounts.csv')
subscriptions = pd.read_csv('ravenstack_subscriptions.csv')

accounts['signup_date'] = pd.to_datetime(accounts['signup_date'])
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'])

# 1~4. 지표 계산
active_paid_mask = subscriptions['end_date'].isna() & (subscriptions['is_trial'] == False)
current_paid_mrr = subscriptions.loc[active_paid_mask, 'mrr_amount'].sum()
subscription_churn_rate = subscriptions['churn_flag'].mean()
cumulative_accounts = accounts['account_id'].nunique()

# 6. 지표 분류를 딕셔너리(=코드)로 관리
metric_classification = {
    "현재 유료 구독 MRR": {
        "변수명": "current_paid_mrr",
        "값": f"${current_paid_mrr:,.0f}",
        "분류": "KPI",
        "이유": "비즈니스 목표(반복 매출 증대)를 화폐 단위로 직접 측정하는 결과 지표이기 때문"
    },
    "전체 구독 이탈률": {
        "변수명": "subscription_churn_rate",
        "값": f"{subscription_churn_rate:.2%}",
        "분류": "보조 지표",
        "이유": "MRR이 왜 늘거나 줄었는지, 매출이 지속 가능한지를 설명해주는 진단/가드레일 지표이기 때문"
    },
    "누적 가입 고객사 수": {
        "변수명": "cumulative_accounts",
        "값": f"{cumulative_accounts:,}개",
        "분류": "허무 지표 후보",
        "이유": "이탈 여부와 무관하게 계속 누적만 되고, 현재 매출 목표와 직접 연결되지 않기 때문"
    },
}

# 표 형태 출력
print(f"{'지표명':<18}{'변수명':<26}{'값':<15}{'분류'}")
for metric_name, info in metric_classification.items():
    print(f"{metric_name:<18}{info['변수명']:<26}{info['값']:<15}{info['분류']}")

# 이유 상세 출력
print("\n분류 이유 상세")
for metric_name, info in metric_classification.items():
    print(f"\n[{info['분류']}] {metric_name} ({info['변수명']} = {info['값']})")
    print(f"  → {info['이유']}")

지표명               변수명                       값              분류
현재 유료 구독 MRR      current_paid_mrr          $10,159,608    KPI
전체 구독 이탈률         subscription_churn_rate   9.72%          보조 지표
누적 가입 고객사 수       cumulative_accounts       500개           허무 지표 후보

분류 이유 상세

[KPI] 현재 유료 구독 MRR (current_paid_mrr = $10,159,608)
  → 비즈니스 목표(반복 매출 증대)를 화폐 단위로 직접 측정하는 결과 지표이기 때문

[보조 지표] 전체 구독 이탈률 (subscription_churn_rate = 9.72%)
  → MRR이 왜 늘거나 줄었는지, 매출이 지속 가능한지를 설명해주는 진단/가드레일 지표이기 때문

[허무 지표 후보] 누적 가입 고객사 수 (cumulative_accounts = 500개)
  → 이탈 여부와 무관하게 계속 누적만 되고, 현재 매출 목표와 직접 연결되지 않기 때문


### 필수 1 답변 작성란

**Q1.** 현재 유료 구독 MRR을 KPI로 볼 수 있는 이유는 무엇인가요?  
- 현재 활성 유료 구독이 매달 만들어내는 반복 매출을 직접 나타내므로 Ravenstcak의 핵심 목표와 일치

**Q2.** 전체 구독 이탈률은 현재 유료 구독 MRR을 이해하는 데 어떻게 도움을 주나요?  
- 이탈율이 높아지면 유료 구독과 반복 매출이 감소할 가능성이 있다

**Q3.** 누적 가입 고객사 수만으로 현재 서비스가 성장한다고 판단하기 어려운 이유는 무엇인가요?
- 누적 가입 고객사 수는 이미 이탈했거나, 구독을 실제로 진행하지 않는 고객사들도 포함될 수 있다.

---

## 필수 2. 요금제별 핵심 지표 비교하기

### 문제 2-1. 어느 요금제를 우선적으로 살펴봐야 하는가?

#### 문제 설명

전체 평균만 확인하면 요금제별 차이를 놓칠 수 있습니다. `Basic`, `Pro`, `Enterprise` 요금제별로 구독 규모, 이탈률, 현재 유료 구독 MRR을 비교하세요.

#### 요구사항

1. `subscriptions`에 `is_active_paid` 컬럼을 만드세요.
   - `end_date`가 비어 있고 `is_trial`이 `False`이면 `True`
2. `plan_tier`별로 다음 값을 집계하여 `plan_metrics`를 만드세요.
   - 전체 구독 수
   - 이탈 구독 수
   - 구독 이탈률
   - 현재 활성 유료 구독 수
3. 현재 활성 유료 구독만 사용해 요금제별 MRR 합계를 계산하고 `active_mrr` 컬럼으로 추가하세요.
4. 이탈률은 백분율로, MRR은 천 단위 구분 기호를 사용하여 출력하세요.
5. 현재 유료 구독 MRR이 가장 큰 요금제와 이탈률이 가장 높은 요금제를 확인하세요.
6. 현재 목표를 고려하여 우선적으로 점검할 요금제 하나를 정하고, 데이터 근거와 확인할 개선 방향을 설명하세요.

#### 해석 질문

**Q1.** 요금제별 이탈률을 비교하는 것이 전체 이탈률만 확인하는 것보다 행동 가능성이 높은 이유는 무엇인가요?  
**Q2.** 현재 유료 구독 MRR이 가장 큰 요금제는 무엇인가요?  
**Q3.** 이탈률이 가장 높은 요금제는 무엇인가요?  
**Q4.** 위 두 결과를 함께 보면 어떤 요금제를 우선적으로 점검할 수 있으며, 그 이유는 무엇인가요?

#### 제출 결과

- `plan_metrics` 집계 코드와 결과
- MRR 및 이탈률 기준 요금제 비교
- 우선 점검 대상과 개선 방향
- Q1~Q4 답변

In [8]:
# 필수 2 코드를 작성하세요.
# 1. is_active_paid 컬럼 생성
subscriptions['is_active_paid'] = subscriptions['end_date'].isna() & (subscriptions['is_trial'] == False)

# 2. plan_tier별 집계
plan_metrics = subscriptions.groupby('plan_tier').agg(
    total_subs=('subscription_id', 'count'),      # 전체 구독 수
    churned_subs=('churn_flag', 'sum'),            # 이탈 구독 수
    churn_rate=('churn_flag', 'mean'),             # 구독 이탈률
    active_paid_subs=('is_active_paid', 'sum')     # 현재 활성 유료 구독 수
)

# 3. 요금제별 active MRR 합계 추가
active_mrr = subscriptions.loc[subscriptions['is_active_paid']].groupby('plan_tier')['mrr_amount'].sum()
plan_metrics['active_mrr'] = active_mrr
plan_metrics = plan_metrics.reset_index()

display_df = plan_metrics.copy()
display_df['churn_rate'] = display_df['churn_rate'].apply(lambda x: f"{x:.2%}")
display_df['active_mrr'] = display_df['active_mrr'].apply(lambda x: f"${x:,.0f}")
print(display_df.to_string(index=False))

 plan_tier  total_subs  churned_subs churn_rate  active_paid_subs active_mrr
     Basic        1602           152      9.49%              1228   $687,914
Enterprise        1723           172      9.98%              1304 $7,546,876
       Pro        1675           162      9.67%              1282 $1,924,818


In [9]:
top_mrr_plan = plan_metrics.loc[plan_metrics['active_mrr'].idxmax(), 'plan_tier']
top_churn_plan = plan_metrics.loc[plan_metrics['churn_rate'].idxmax(), 'plan_tier']

print(top_mrr_plan)
print(top_churn_plan)

Enterprise
Enterprise


### 필수 2 답변 작성란

**Q1.** 요금제별 이탈률을 비교하는 것이 전체 이탈률만 확인하는 것보다 행동 가능성이 높은 이유는 무엇인가요?  
- 전체 이탈률은 서비스 전체의 현상을 보여주고, 요금제별 이탈률은 어느 요금제를 먼저 점검해야하는지를 판단한다.
- 다음 회의가 열린다면 담당 팀이 개선 대상을 명확하게 설정하여 개선을 진행할 수 있다.

**Q2.** 현재 유료 구독 MRR이 가장 큰 요금제는 무엇인가요?  
- Enterprise, $7,546,876.

**Q3.** 이탈률이 가장 높은 요금제는 무엇인가요?  
- Enterprise, 9.98%

**Q4.** 위 두 결과를 함께 보면 어떤 요금제를 우선적으로 점검할 수 있으며, 그 이유는 무엇인가요?
- Enterprise를 우선적으로 점검한다.
- 현재 MRR기여도가 가장 크면서 이탈률이 가장 높아, 이 그룹의 이탈률을 줄이는 것이 다음달 매출 방어 효과가 가장 크다.

---

## 과제 1. 평가 문항 기반 독립 과제

### 문제 3-1. 유입 경로별 고객사 이탈률 비교하기

#### 문제 설명

Ravenstack 마케팅팀은 고객사가 유입된 경로에 따라 이탈 수준이 다른지 확인하려고 합니다. 유입 경로별 고객사 수와 이탈률을 비교하여 우선 점검할 유입 경로를 찾으세요.

> 이 과제는 필수 2에서 수행한 그룹별 지표 집계와 해석을 새로운 컬럼에 동일하게 적용하는 문제입니다.

#### 요구사항

1. `accounts`를 `referral_source`별로 그룹화하세요.
2. 다음 값을 집계하여 `source_metrics`를 만드세요.
   - 전체 고객사 수
   - 이탈 고객사 수
   - 고객사 이탈률
3. 고객사 이탈률이 높은 순서로 정렬하세요.
4. 이탈률은 소수점 둘째 자리의 백분율로 출력하세요.
5. 이탈률이 가장 높은 유입 경로를 찾으세요.
6. 해당 유입 경로를 우선 점검 대상으로 정하고, 확인할 개선 방향을 한 가지 제안하세요.
7. 유입 경로별 이탈률을 좋은 지표의 세 조건으로 평가하세요.
   - 행동 가능성
   - 비교 가능성
   - 이해 용이성

#### 해석 질문

**Q1.** 고객사 이탈률이 가장 높은 유입 경로는 무엇인가요?  
**Q2.** 유입 경로별 이탈률은 마케팅팀의 행동으로 어떻게 연결할 수 있나요?  
**Q3.** 유입 경로별 이탈률은 좋은 지표의 세 조건을 충족하나요?
**Q4.** KPI·보조 지표·허무 지표는 어떻게 다른가요? 고객사 이탈을 줄이려는 목적에서 각각의 지표 예시와 선정 이유를 제시하세요. 허무 지표는 어떤 맥락에서 성과 판단에 도움이 되지 않는지도 설명하세요.

#### 제출 결과

- `source_metrics` 집계 코드와 결과
- 우선 점검할 유입 경로
- 개선 방향
- 좋은 지표의 조건 평가
- Q1~Q4 답변

In [13]:
# 과제 1 코드를 작성하세요.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ====================================================
# 1~2. accounts를 referral_source별로 그룹화 및 집계 (source_metrics)
# ====================================================
# referral_source별 전체 고객사 수, 이탈 고객사 수 집계
source_metrics = accounts.groupby("referral_source").agg(
    total_accounts=("account_id", "count"),
    churned_accounts=("churn_flag", "sum"),
)
print(source_metrics)

# 고객사 이탈률 계산 (이탈 고객사 수 ÷ 전체 고객사 수)
source_metrics["churn_rate"] = (
    source_metrics["churned_accounts"] / source_metrics["total_accounts"]
)

# ====================================================
# 3. 고객사 이탈률이 높은 순서로 정렬
# ====================================================
source_metrics = source_metrics.sort_values(by="churn_rate", ascending=False)
print(source_metrics)

# ====================================================
# 4. 소수점 둘째 자리 백분율 포맷팅 출력
# ====================================================
print("=== 유입 경로별 고객사 이탈률 (source_metrics) ===")
display_metrics = source_metrics.copy()
display_metrics["total_accounts"] = display_metrics["total_accounts"].map(
    lambda x: f"{x:,}개"
)
display_metrics["churned_accounts"] = display_metrics["churned_accounts"].map(
    lambda x: f"{x:,}개"
)
display_metrics["churn_rate"] = display_metrics["churn_rate"].map(
    lambda x: f"{x * 100:.2f}%"
)

display(display_metrics)

# ====================================================
# 5. 이탈률이 가장 높은 유입 경로 추출
# ====================================================
highest_source = source_metrics.index[0]
highest_rate = source_metrics.loc[highest_source, "churn_rate"]
print(
    f"\n[이탈률 최고 유입 경로]: {highest_source} ({highest_rate * 100:.2f}%)"
)

                 total_accounts  churned_accounts
referral_source                                  
ads                          98                23
event                        96                29
organic                     114                20
other                       103                25
partner                      89                13
                 total_accounts  churned_accounts  churn_rate
referral_source                                              
event                        96                29    0.302083
other                       103                25    0.242718
ads                          98                23    0.234694
organic                     114                20    0.175439
partner                      89                13    0.146067
=== 유입 경로별 고객사 이탈률 (source_metrics) ===


,total_accounts,churned_accounts,churn_rate
referral_source,,,
event,96개,29개,30.21%
other,103개,25개,24.27%
ads,98개,23개,23.47%
organic,114개,20개,17.54%
partner,89개,13개,14.61%



[이탈률 최고 유입 경로]: event (30.21%)


### 과제 1 답변 작성란

**Q1.** 고객사 이탈률이 가장 높은 유입 경로는 무엇인가요?  
- event (30.21%)

**Q2.** 유입 경로별 이탈률은 마케팅팀의 행동으로 어떻게 연결할 수 있나요?  
- 단순히 신규 가입자수를 늘리는 것 보다는 이탈률이 낮은 채널에 예산을 집중하는 방향으로 마케팅 전략 변경

**Q3.** 유입 경로별 이탈률은 좋은 지표의 세 조건을 충족하나요?
- 네, 비율 지표로서 규모가 다른 채널 간 비교가 용이하고(비교 가능성), 직관적인 백분율로 구성되어 누구나 이해하기 쉬우며(이해 용이성), 예산 삭감/증액 및 캠페인 타깃 조정이라는 즉각적인 의사결정이 가능하기 때문(행동 가능성).

**Q4.** KPI·보조 지표·허무 지표는 어떻게 다른가요? 고객사 이탈을 줄이려는 목적에서 각각의 지표 예시와 선정 이유를 제시하세요. 허무 지표는 어떤 맥락에서 성과 판단에 도움이 되지 않는지도 설명하세요.
- KPI는 비즈니스의 성과를 직접적으로 보여주는 핵심 지표이고, 보조지표는 KPI의 원인이 되거나 설명할 수 있는 지표, 허무지표는 수치로는 그럴듯하나 실제 비즈니스 개선에 도움이 되지 않는 지표를 뜻한다. 
- 고객사 이탈을 줄이려는 목적에서 각각의 지표 예시와 선정 이유를 제시: 
- KPI: 이탈률/유지율 -> 구독상품 위주이므로 고객사가 이탈하지 않고 계속 이용하는 것이 중요하기 때문이다.
- 보조지표: 기능사용률/유료전환율 -> 신규 가입 후 유료 구독으로 전환하는 비율이 KPI에 큰 영향을 미치기 떄문이다.
- 허무지표: 누적방문자수/신규가입자수 -> 방문자 수가 아무리 늘어나도 제품을 지속해서 쓰는 유효 고객이 늘어나지 않는다면 이탈 방지에 아무런 도움이 되지 않기 때문이다.


---

## 실습 마무리

아래 질문에 답하세요.

1. 이번 실습에서 해결하려고 한 비즈니스 문제는 무엇인가요?
- 유료 구독을 안정적으로 유지하면서 반복 매출을 늘리기 위한 핵심 지표를 관리하고, 어느 대상을 우선적으로 개선해야하는지 확인했다.

2. 현재 목표를 직접 보여주는 KPI로 어떤 지표를 선택했나요?
- 현재 활성화된 유료 구독자들이 만드는 매출을 MRR로 선택했다.

3. KPI의 변화를 이해하기 위해 어떤 보조 지표를 확인했나요?
- 전체 구독 이탈률과 요금제별 이탈률을 확인했다.
- 이탈률은 현재 또는 미래의 매출 감소로 이어지기 때문이다.

4. 허무 지표 후보를 핵심 성과로 사용할 때 어떤 문제가 생길 수 있나요?
- 누적 가입 고객사를 핵심 성과로 사용한다면 현재 구독을 하지 않거나, 활동하지 않아도
- 실제 성과를 과대평가할 수 없다.

5. 어떤 분석 결과를 근거로 개선 대상을 정했나요?
- 요금제별로 MRR과 이탈률을 분석하여 Enterprise를 우선 점검 대상으로 정했다.